# 22.7 ML 的 CI/CD:自动化测试与发布 / CI/CD for ML

**中文**:软件工程有个铁律:**每次改代码,都要自动跑一遍"检查+测试+构建+部署"**——这就是 **CI/CD(持续集成/持续交付)**。它的价值是:让"坏改动"在**到达生产之前**就被自动拦下,而不是等用户报障才发现。但机器学习的 CI/CD 有一个**独特的难点**:软件只需测"代码对不对",而 ML 还要测"**数据对不对、模型好不好**"——一个能跑通、没语法错的代码,可能训出一个**准确率暴跌的烂模型**,照样是灾难。所以 ML 的 CI/CD 多了两道关键防线:**数据校验** 和 **模型质量门(quality gate)**。本节从零实现一个带质量门的 CI/CD 流水线,亲眼看它**拦下一个退化的模型**,再给出真实的 GitHub Actions 配置。
**English**: Software engineering has an iron rule: **every code change automatically runs "check + test + build + deploy"** — that's **CI/CD (Continuous Integration/Continuous Delivery)**. Its value: catching "bad changes" automatically **before they reach production**, rather than waiting for user complaints. But ML's CI/CD has a **unique difficulty**: software only tests "is the code correct," while ML must also test "**is the data correct, is the model good**" — code that runs without syntax errors can still train a **model with crashed accuracy**, a disaster all the same. So ML's CI/CD adds two key defenses: **data validation** and a **model quality gate**. This section builds a CI/CD pipeline with a quality gate from scratch, watches it **block a degraded model**, then gives real GitHub Actions config.

---

**中文**:**CI / CD / CT 三个概念**:
**English**: **Three concepts: CI / CD / CT**:
- **中文**:**CI(持续集成)**:每次 push 代码,自动跑代码检查(lint)、单元测试、**数据校验**、训练、评估。目标:**尽早发现问题**。
  **CI (Continuous Integration)**: every code push auto-runs lint, unit tests, **data validation**, training, evaluation. Goal: **catch problems early**.
- **中文**:**CD(持续交付/部署)**:通过所有检查后,自动把新模型/服务打包、发布到(测试/生产)环境。目标:**发布快速可靠、可回滚**。
  **CD (Continuous Delivery/Deployment)**: after all checks pass, automatically package and release the new model/service to (staging/production). Goal: **fast, reliable, rollback-able releases**.
- **中文**:**CT(持续训练)** —— ML 独有:数据会随时间变化(漂移,见 22.10),所以要**自动地用新数据重新训练**,并让新模型走同一套 CI/CD 关卡。
  **CT (Continuous Training)** — ML-unique: data changes over time (drift, see 22.10), so **automatically retrain on new data** and put the new model through the same CI/CD gates.

**中文**:**ML 特有的两道关卡**(普通软件 CI 没有):
**English**: **Two ML-specific gates** (absent in ordinary software CI):
- **中文**:**数据校验(data validation)**:训练前检查数据的 schema、缺失、取值范围、分布是否符合预期——**烂数据训不出好模型(garbage in, garbage out)**,得在训练前拦下。
  **Data validation**: before training, check the data's schema, missing values, value ranges, and whether the distribution matches expectations — **bad data trains bad models (garbage in, garbage out)**, so block it before training.
- **中文**:**模型质量门(quality gate)**:训完的**候选模型必须在留出集上达到/超过当前生产模型的水平,才允许发布**。这是防止"悄悄发布了一个更差的模型"的最后一道防线——代码能跑 ≠ 模型变好。
  **Model quality gate**: the trained **candidate model must reach/exceed the current production model's level on a holdout set before release is allowed**. This is the last defense against "silently deploying a worse model" — code running ≠ model improved.

> 💡 **面试速查 / Interview cheat-sheet（★★ MLOps 必考）**
> **中文**:**CI/CD**=每次改动自动 lint→测试→构建→部署, 让坏改动在到生产前被拦。**ML 特有**:除了测代码, 还要**测数据(data validation: schema/缺失/范围/分布)+ 测模型(quality gate: 候选模型须≥生产基线才发布)**——因为"代码能跑≠模型变好"。**CT(持续训练)**:数据漂移→自动用新数据重训并过同一关卡。**关键实践**:①**质量门**(留出集/影子评估, 退化就 BLOCK, 防悄悄发烂模型)②**数据/模型版本化**(DVC/lakeFS + 模型registry, 可复现可回滚)③**渐进发布**(金丝雀/影子/蓝绿, 小流量验证再全量, 接 22.11)④测试金字塔(单元测试代码 + 数据测试 + 模型行为测试如"输入涨价预测应涨")。**工具**:GitHub Actions/GitLab CI(编排)、**CML**(在 PR 里贴模型指标对比)、**DVC**(数据/模型版本+流水线)、MLflow(22.8)。**vs 普通软件 CI**:多了数据和模型两个"会变"的维度。面试金句:*"ML 的 CI/CD 除了 lint/单元测试, 还要加数据校验和模型质量门——候选模型在留出集上不达到生产基线就阻断发布, 防止代码能跑但模型退化悄悄上线; 配合数据/模型版本化(DVC)、持续训练(数据漂移触发重训)、金丝雀渐进发布和可回滚; 用 GitHub Actions+CML+DVC 落地。"*
> **English**: **CI/CD** = every change auto-runs lint→test→build→deploy, blocking bad changes before production. **ML-specific**: beyond testing code, also **test data (data validation: schema/missing/ranges/distribution) + test the model (quality gate: candidate must ≥ production baseline to release)** — because "code runs ≠ model improved." **CT (Continuous Training)**: data drift → auto-retrain on new data through the same gates. **Key practices**: ① **quality gate** (holdout/shadow evaluation, BLOCK on regression, prevent silently shipping a worse model) ② **data/model versioning** (DVC/lakeFS + model registry, reproducible and rollback-able) ③ **progressive rollout** (canary/shadow/blue-green, validate on small traffic then full, ties to 22.11) ④ testing pyramid (unit tests for code + data tests + model behavior tests like "raise price → prediction should rise"). **Tools**: GitHub Actions/GitLab CI (orchestration), **CML** (post model metric comparisons in PRs), **DVC** (data/model versioning + pipelines), MLflow (22.8). **vs ordinary software CI**: adds two "changing" dimensions — data and model. Interview line: *"ML's CI/CD adds data validation and a model quality gate beyond lint/unit tests — a candidate model that doesn't reach the production baseline on a holdout blocks release, preventing a runnable-but-degraded model from silently shipping; combine with data/model versioning (DVC), continuous training (drift triggers retrain), canary progressive rollout, and rollback; implement with GitHub Actions + CML + DVC."*


In [ ]:

# ============================================================
# 从零实现带"模型质量门"的 CI/CD 流水线 / CI/CD pipeline with a model quality gate
# 中文:各阶段依次执行, 失败即停(fail-fast)。关键是最后的"质量门":候选模型必须达到生产基线才放行。
# English: stages run in order, stop on first failure (fail-fast). The key is the final "quality gate": the candidate
#      must reach the production baseline to be released.
# ============================================================
import numpy as np, warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
np.random.seed(0)
X,y=make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=0)
Xtr,Xte,ytr,yte=train_test_split(X, y, test_size=0.3, random_state=0)
PROD_BASELINE=0.88                                         # 当前生产模型的准确率 / current production model's accuracy
state={}

def run_pipeline(stages):                                  # 流水线执行器:失败即停 / pipeline runner: fail-fast
    for name, fn in stages:
        ok, msg = fn()
        print(f"  [{'PASS' if ok else 'FAIL'}] {name:20} {msg}")
        if not ok:
            print(f"  ✗ 流水线在「{name}」失败 → 停止, 不部署 (fail-fast)"); return False
    print("  ✓ 全部通过 → 发布新模型到生产"); return True

def lint():            return True, "代码风格无误 / no style errors"
def unit_tests():      return True, "12 个单元测试通过 / 12 tests passed"
def data_validation(): return (not np.isnan(X).any() and X.shape[1]==20), "schema OK, 无缺失, 范围正常"  # 数据校验 / data check
def train():           state["cand"]=RandomForestClassifier(n_estimators=100,random_state=0).fit(Xtr,ytr); return True,"候选模型已训练"
def quality_gate():                                        # ★ 模型质量门 / the model quality gate
    acc=accuracy_score(yte, state["cand"].predict(Xte)); state["acc"]=acc
    ok = acc >= PROD_BASELINE
    return ok, f"候选 acc={acc:.3f} vs 生产基线 {PROD_BASELINE} → {'达标, 放行' if ok else '退化, 阻断!'}"

print("=== 运行 #1:一个好模型 ===")
run_pipeline([("lint",lint),("unit tests",unit_tests),("data validation",data_validation),
              ("train model",train),("quality gate",quality_gate)])

print("\n=== 运行 #2:一个退化的候选模型撞上质量门(代码能跑, 但模型变差)===")
def train_weak(): state["cand"]=LogisticRegression(max_iter=5).fit(Xtr[:50],ytr[:50]); return True,"弱候选模型已训练(欠拟合)"
run_pipeline([("lint",lint),("unit tests",unit_tests),("data validation",data_validation),
              ("train model",train_weak),("quality gate",quality_gate)])


**中文**:上面从零实现了流水线。下面是真实的 **GitHub Actions** 配置(每次 push/PR 自动触发):
**English**: The above implements the pipeline from scratch. Below is real **GitHub Actions** config (auto-triggered on every push/PR):

```yaml
# .github/workflows/ml-ci.yml —— push/PR 时自动运行 / runs automatically on push/PR
name: ML CI/CD
on: [push, pull_request]

jobs:
  build-test-gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -r requirements.txt

      - name: Lint            # 代码风格 / code style
        run: ruff check .
      - name: Unit tests      # 代码正确性 / code correctness
        run: pytest tests/ -v
      - name: Data validation # ★ ML 特有:校验数据 schema/缺失/范围 / validate data
        run: python scripts/validate_data.py
      - name: Train model     # 训练候选模型 / train candidate
        run: python scripts/train.py --out model.joblib
      - name: Quality gate    # ★ ML 特有:候选模型不达生产基线就让这一步失败→阻断发布 / block release if below baseline
        run: python scripts/quality_gate.py --model model.joblib --baseline 0.88

  deploy:
    needs: build-test-gate    # 只有上面全过了才部署(接 22.5/22.6)/ deploy only if all checks passed
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - run: echo "docker build & push, then kubectl set image ..."   # 构建镜像 → 滚动更新 K8s
```
**中文**:`quality_gate.py` 里如果 `acc < baseline` 就 `sys.exit(1)`,这一步失败,`deploy` 因 `needs` 依赖而不会执行——**退化的模型被自动挡在生产之外**。
**English**: In `quality_gate.py`, if `acc < baseline` then `sys.exit(1)`; this step fails and `deploy` (via `needs`) won't run — **the degraded model is automatically kept out of production**.


In [ ]:

# ============================================================
# 可视化:CI/CD 流水线(质量门拦截退化模型)/ CI/CD pipeline (quality gate blocks a degraded model)
# ============================================================
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
stages=["lint","unit tests","data\nvalidation","train","quality\ngate","deploy"]
for col,(title,fail_at) in enumerate([("运行#1:好模型 → 全绿, 部署", None), ("运行#2:退化模型 → 质量门拦截", 4)]):
    a=ax[col]; a.axis("off"); a.set_title(title,fontsize=12,weight="bold")
    for i,s in enumerate(stages):
        if fail_at is None: c="#55A868"          # all pass
        elif i<fail_at:     c="#55A868"          # passed before gate
        elif i==fail_at:    c="#C44E52"          # gate FAILS
        else:               c="#CCCCCC"          # never reached
        a.add_patch(plt.Rectangle((0.05,0.75-i*0.13),0.9,0.1,fc=c,alpha=0.5,ec=c,transform=a.transAxes))
        tag="✓" if c=="#55A868" else ("✗ 阻断" if c=="#C44E52" else "跳过")
        a.text(0.5,0.8-i*0.13,f"{s}   [{tag}]",ha="center",va="center",fontsize=9,transform=a.transAxes)
        if i<len(stages)-1: a.annotate("",xy=(0.5,0.75-i*0.13),xytext=(0.5,0.75-i*0.13+0.02),arrowprops=dict(arrowstyle="->"),transform=a.transAxes)
    a.text(0.5,0.02,"退化模型在'质量门'被拦, deploy 从不执行" if fail_at else "全部通过, 自动部署到生产",ha="center",fontsize=8,style="italic",transform=a.transAxes)
plt.tight_layout(); plt.savefig("/tmp/mlops07_viz.png",dpi=80); plt.show()
print("质量门是 ML CI/CD 的灵魂:它保证'只有真正更好的模型才能上线', 而不是'代码能跑就上线'")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **ML 的 CI/CD 比普通软件多一个致命维度:代码能跑 ≠ 模型变好**:普通软件 CI 的逻辑是"测试通过 = 可以发布",因为软件的行为完全由代码决定。但 ML 不是——同一份**毫无 bug、测试全绿**的训练代码,可能因为数据变了、超参调坏了、特征漏了,训出一个准确率暴跌的模型。我们的实验里,那个欠拟合的候选模型代码**跑得好好的、语法零错误**,但准确率只有 0.70,远低于生产的 0.88。如果没有质量门,它会**堂而皇之地通过所有传统 CI 检查、被自动部署到生产**,悄无声息地让业务指标下滑。**模型质量门就是那道"代码测试管不了、专为 ML 而设"的防线**——这是 ML CI/CD 区别于软件 CI/CD 的本质。
2. **"自动化"的真正价值是把判断力固化成不可绕过的关卡**:一个有经验的工程师当然知道"发布前要检查模型有没有变差"。但人会忘、会赶时间、会在周五下午图省事跳过检查。CI/CD 的意义,是把这些"应该做的检查"**变成机器自动执行、无法绕过的门**:每次改动都跑、不达标就红、红了就不许合并/部署。这把"依赖个人自觉"升级成"系统性保证"。对 ML 尤其重要,因为 ML 的失败往往是**静默的**(模型不会崩溃报错,只是预测悄悄变烂),没有自动化的质量门和监控(22.10),你可能几周后才从业务数字里发现。
3. **诚实的复杂性:ML 的 CI/CD 是"可复现性"的系统工程,远不止跑个 pytest**。①**可复现是前提**:要让质量门有意义,每次训练必须可复现(固定随机种子、钉住数据版本和依赖版本——接 22.1),否则"这次 0.87 下次 0.89"你根本不知道是模型变了还是随机波动。这就需要**数据版本化(DVC/lakeFS)+ 模型版本化(registry)+ 环境固化(容器)** 三件套。②**测什么是门学问**:除了传统单元测试,ML 还要测**数据**(schema、分布、缺失——用 Great Expectations/pandera)、测**模型行为**(不变性测试如"输入微扰预测应稳定"、方向性测试如"涨价→购买概率应下降")。③**质量门的阈值怎么定**是权衡:太松形同虚设,太紧则正常的模型波动也被拦、发布寸步难行;实践中常用"不差于基线一定容差"或影子/金丝雀评估。④**持续训练(CT)** 引入了新循环:数据漂移触发重训,但重训的模型必须走同一套关卡才能替换生产——否则"自动重训"反而成了"自动上线烂模型"的通道。**结论:ML 的 CI/CD 核心是用自动化的数据校验 + 模型质量门,把'只有真正更好且可复现的模型才能上生产'变成不可绕过的系统保证;它建立在数据/模型/环境版本化的可复现地基上,是 MLOps 从'手工发模型'走向'工程化可信交付'的关键。**

**English**:
1. **ML's CI/CD has one fatal extra dimension: code running ≠ model improved**: ordinary software CI's logic is "tests pass = ready to release," because software behavior is fully determined by code. But ML isn't — the same **bug-free, all-green** training code can train a crashed-accuracy model because data changed, hyperparameters were mis-tuned, or a feature was dropped. In our experiment, the underfit candidate's code **ran fine with zero syntax errors** but reached only 0.70 accuracy, far below production's 0.88. Without a quality gate, it would **openly pass all traditional CI checks and be auto-deployed to production**, silently degrading business metrics. **The model quality gate is exactly the defense "code tests can't cover, designed for ML"** — the essence distinguishing ML CI/CD from software CI/CD.
2. **"Automation's" real value is hardening judgment into an unbypassable gate**: an experienced engineer of course knows "check whether the model got worse before releasing." But people forget, rush, and skip checks on a Friday afternoon for convenience. CI/CD's point is turning these "checks you should do" into **machine-executed, unbypassable gates**: every change runs them, below-threshold goes red, red blocks merge/deploy. This upgrades "rely on personal diligence" to "systematic guarantee." Especially important for ML, whose failures are often **silent** (the model doesn't crash, predictions just quietly worsen); without an automated quality gate and monitoring (22.10), you might discover it weeks later from business numbers.
3. **Honest complexity: ML's CI/CD is systems engineering for "reproducibility," far more than running pytest**. ① **Reproducibility is a prerequisite**: for the quality gate to mean anything, each training must be reproducible (fixed random seeds, pinned data and dependency versions — per 22.1), else "0.87 this time, 0.89 next" leaves you unsure whether the model changed or it's just random variation. This needs the trio of **data versioning (DVC/lakeFS) + model versioning (registry) + environment freezing (containers)**. ② **What to test is an art**: beyond traditional unit tests, ML must test **data** (schema, distribution, missing — via Great Expectations/pandera) and **model behavior** (invariance tests like "small input perturbation → stable prediction", directional tests like "raise price → purchase probability should drop"). ③ **Setting the gate threshold** is a tradeoff: too loose is meaningless, too tight blocks normal model variation and stalls releases; in practice use "no worse than baseline by some tolerance" or shadow/canary evaluation. ④ **Continuous training (CT)** adds a new loop: drift triggers retraining, but the retrained model must pass the same gates to replace production — else "auto-retrain" becomes "auto-ship a bad model." **Conclusion: ML's CI/CD centers on automated data validation + a model quality gate, turning "only a genuinely better and reproducible model reaches production" into an unbypassable systematic guarantee; it rests on the reproducibility foundation of data/model/environment versioning, and is key to MLOps moving from "manually shipping models" to "engineered, trustworthy delivery."**

> 💼 **实战视角 / Practical angle**
> **中文**:ML CI/CD 落地:①**流水线**(GitHub Actions/GitLab CI):lint→单元测试→**数据校验**(Great Expectations/pandera)→训练→**质量门**(候选≥基线才发)→构建镜像(22.5)→部署(22.6);②**质量门**用留出集或影子评估, 退化就 `exit(1)` 阻断;③**版本化**:代码(git)+ 数据(DVC/lakeFS)+ 模型(MLflow registry, 22.8)——可复现可回滚;④**CML** 自动在 PR 里贴出候选 vs 生产的指标对比图, 让 review 者一眼看到影响;⑤**渐进发布**:金丝雀/影子(小流量先验证, 接 22.11), 可快速回滚;⑥**持续训练**:调度(Airflow/Argo)定期/漂移触发重训, 走同一关卡。**别做的**:没有质量门就自动部署、训练不可复现、直接全量发布新模型。面试金句:*"ML CI/CD 在 lint/测试之外加数据校验和模型质量门——候选模型在留出集不达生产基线就阻断发布, 防止代码能跑但模型退化悄悄上线; 要做数据/模型/环境版本化保证可复现可回滚, 用金丝雀渐进发布, 持续训练也走同一关卡; 工具是 GitHub Actions+DVC+CML+MLflow。"*
> **English**: ML CI/CD in practice: ① **pipeline** (GitHub Actions/GitLab CI): lint → unit tests → **data validation** (Great Expectations/pandera) → training → **quality gate** (candidate ≥ baseline to release) → build image (22.5) → deploy (22.6); ② **quality gate** uses a holdout or shadow evaluation, `exit(1)` to block on regression; ③ **versioning**: code (git) + data (DVC/lakeFS) + model (MLflow registry, 22.8) — reproducible and rollback-able; ④ **CML** auto-posts candidate-vs-production metric comparison charts in the PR so reviewers see the impact at a glance; ⑤ **progressive rollout**: canary/shadow (validate on small traffic first, ties to 22.11), fast rollback; ⑥ **continuous training**: schedule (Airflow/Argo) periodic/drift-triggered retraining through the same gates. **Don't**: auto-deploy without a quality gate, non-reproducible training, or full-rollout of a new model directly. Interview line: *"ML CI/CD adds data validation and a model quality gate beyond lint/tests — a candidate model that doesn't reach the production baseline on a holdout blocks release, preventing a runnable-but-degraded model from silently shipping; do data/model/environment versioning for reproducibility and rollback, use canary progressive rollout, and put continuous training through the same gates; tools are GitHub Actions + DVC + CML + MLflow."*

---
### 小结 / Summary
- **中文**:CI/CD=每次改动自动 lint→测试→构建→部署; ML 特有加数据校验 + 模型质量门(候选须≥生产基线才发)。
- **English**: CI/CD = every change auto lint→test→build→deploy; ML adds data validation + a model quality gate (candidate must ≥ production baseline to release).
- **中文**:代码能跑≠模型变好——质量门拦下退化模型(实验:0.70 被 0.88 基线阻断), 是 ML CI/CD 的灵魂。
- **English**: Code running ≠ model improved — the quality gate blocks degraded models (experiment: 0.70 blocked by 0.88 baseline), the soul of ML CI/CD.
- **中文**:建立在数据/模型/环境版本化的可复现地基上; 配持续训练、金丝雀渐进发布、可回滚; 工具 GitHub Actions+DVC+CML+MLflow。
- **English**: Built on the reproducibility foundation of data/model/environment versioning; with continuous training, canary progressive rollout, rollback; tools GitHub Actions + DVC + CML + MLflow.
